# XAI (Explainable AI) for Transformer-based RUL Prediction

This notebook implements comprehensive explainability techniques for the transformer-based remaining useful life (RUL) prediction model trained on N-CMAPSS data.

**Techniques covered:**
1. Attention Weight Visualization
2. SHAP Feature Importance
3. Gradient-based Saliency Maps
4. Temporal Importance Analysis
5. Prediction Confidence Analysis

## 1. Setup and Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h5py
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.test.is_built_with_cuda())

ModuleNotFoundError: No module named 'tensorflow'

## 2. Load Data and Model

Load your trained transformer model and test dataset

In [ ]:
# Load N-CMAPSS dataset
# Update path to your dataset
DATA_PATH = "../archive/af/nasa/new_dataset/data_set/N-CMAPSS_DS03-012.h5"

def load_ncmapss_data(filepath, test_split=0.2):
    """Load N-CMAPSS data from HDF5 file"""
    with h5py.File(filepath, 'r') as f:
        W = np.array(f['W'])
        X_s = np.array(f['X_s'])
        X_v = np.array(f['X_v'])
        T = np.array(f['T'])
        Y = np.array(f['Y'])
        A = np.array(f['A'])
    
    # Combine features: working conditions + sensor data
    X = np.concatenate([W, X_s, X_v], axis=-1)
    
    # Normalize
    scaler = StandardScaler()
    X_normalized = scaler.fit_transform(X.reshape(-1, X.shape[-1])).reshape(X.shape)
    
    # Split into train/test
    n_samples = X_normalized.shape[0]
    split_idx = int(n_samples * (1 - test_split))
    
    X_train = X_normalized[:split_idx]
    X_test = X_normalized[split_idx:]
    Y_train = Y[:split_idx]
    Y_test = Y[split_idx:]
    
    return X_train, X_test, Y_train, Y_test, scaler

# Load data
try:
    X_train, X_test, Y_train, Y_test, scaler = load_ncmapss_data(DATA_PATH)
    print(f"Data loaded successfully!")
    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"Y_train shape: {Y_train.shape}")
    print(f"Y_test shape: {Y_test.shape}")
    print(f"Number of features: {X_train.shape[-1]}")
except Exception as e:
    print(f"Error loading data: {e}")
    print("Please update DATA_PATH to point to your N-CMAPSS dataset")

In [ ]:
# Load your trained model
# Update path to your saved model
MODEL_PATH = "./trained_model.h5"  # or .keras

try:
    model = keras.models.load_model(MODEL_PATH)
    print("Model loaded successfully!")
    model.summary()
except Exception as e:
    print(f"Error loading model: {e}")
    print("\nTo generate predictions without a saved model, please provide:")
    print("1. Your model architecture code")
    print("2. Path to saved weights")

## 3. Attention Visualization

Extract and visualize attention weights from transformer layers

In [ ]:
def get_attention_weights(model, input_data):
    """
    Extract attention weights from transformer model
    
    For models with MultiHeadAttention layers, this creates a function model
    that outputs intermediate attention weights.
    """
    # Find attention layers in the model
    attention_layer_names = [layer.name for layer in model.layers 
                            if 'attention' in layer.name.lower() or 'multi_head' in layer.name.lower()]
    
    if not attention_layer_names:
        print("No attention layers found. Checking all layer outputs...")
        for i, layer in enumerate(model.layers):
            print(f"{i}: {layer.name} - {layer.__class__.__name__}")
        return None
    
    print(f"Found attention layers: {attention_layer_names}")
    
    # Create model to extract attention outputs
    intermediate_models = []
    for layer_name in attention_layer_names[:3]:  # Limit to first 3 for clarity
        intermediate_layer = model.get_layer(layer_name)
        intermediate_model = keras.Model(inputs=model.input, outputs=intermediate_layer.output)
        attention_output = intermediate_model.predict(input_data)
        intermediate_models.append(attention_output)
    
    return intermediate_models

# Extract attention from sample test data
sample_idx = 0
sample_input = X_test[sample_idx:sample_idx+1]  # Get single sample with batch dimension
print(f"Sample input shape: {sample_input.shape}")

try:
    attention_outputs = get_attention_weights(model, sample_input)
except Exception as e:
    print(f"Note: {e}")
    print("\nThis is expected if model doesn't have named attention layers.")
    print("For custom attention layers, you may need to modify the extraction method.")

## 4. Gradient-based Saliency Maps

Compute gradients to identify which timesteps and features are most important

In [ ]:
def compute_temporal_saliency(model, input_seq):
    """
    Compute gradient-based saliency to show which timesteps matter most
    Input shape: (batch, timesteps, features)
    Output: (batch, timesteps) - importance scores per timestep
    """
    input_tensor = tf.convert_to_tensor(input_seq, dtype=tf.float32)
    
    with tf.GradientTape() as tape:
        tape.watch(input_tensor)
        predictions = model(input_tensor, training=False)
    
    # Compute gradients
    gradients = tape.gradient(predictions, input_tensor)
    
    if gradients is None:
        print("Warning: Gradients are None. Model may not be differentiable.")
        return None
    
    # Aggregate across feature dimension to get temporal importance
    temporal_saliency = tf.reduce_mean(tf.abs(gradients), axis=-1)
    
    return temporal_saliency.numpy()

def compute_feature_saliency(model, input_seq):
    """
    Compute gradient-based saliency to show which features matter most
    Returns (features,) - importance scores per feature
    """
    input_tensor = tf.convert_to_tensor(input_seq, dtype=tf.float32)
    
    with tf.GradientTape() as tape:
        tape.watch(input_tensor)
        predictions = model(input_tensor, training=False)
    
    gradients = tape.gradient(predictions, input_tensor)
    
    if gradients is None:
        return None
    
    # Aggregate across time dimension to get feature importance
    feature_saliency = tf.reduce_mean(tf.abs(gradients), axis=1)
    
    return feature_saliency.numpy()

# Compute saliency for sample
try:
    temporal_sal = compute_temporal_saliency(model, sample_input)
    feature_sal = compute_feature_saliency(model, sample_input)
    
    if temporal_sal is not None:
        print(f"Temporal saliency shape: {temporal_sal.shape}")
        print(f"Feature saliency shape: {feature_sal.shape}")
    else:
        print("Could not compute saliency - model may need recompilation for gradient tracking")
except Exception as e:
    print(f"Error computing saliency: {e}")

## 5. Visualization: Temporal Importance Heatmap

In [ ]:
# Visualize temporal saliency across multiple samples
n_samples_viz = 10
temporal_saliencies = []

for i in range(n_samples_viz):
    sample = X_test[i:i+1]
    sal = compute_temporal_saliency(model, sample)
    if sal is not None:
        temporal_saliencies.append(sal[0])

if temporal_saliencies:
    temporal_saliency_matrix = np.array(temporal_saliencies)
    
    fig, ax = plt.subplots(figsize=(14, 8))
    im = ax.imshow(temporal_saliency_matrix, aspect='auto', cmap='hot', interpolation='nearest')
    ax.set_xlabel('Timestep', fontsize=12)
    ax.set_ylabel('Sample Index', fontsize=12)
    ax.set_title('Temporal Importance Heatmap - Which timesteps matter for RUL prediction?', fontsize=14)
    plt.colorbar(im, ax=ax, label='Importance Score')
    plt.tight_layout()
    plt.show()
    
    # Show average temporal importance
    avg_temporal_importance = np.mean(temporal_saliency_matrix, axis=0)
    plt.figure(figsize=(14, 5))
    plt.plot(avg_temporal_importance, linewidth=2)
    plt.fill_between(range(len(avg_temporal_importance)), avg_temporal_importance, alpha=0.3)
    plt.xlabel('Timestep', fontsize=12)
    plt.ylabel('Average Importance', fontsize=12)
    plt.title('Average Temporal Importance Across Samples', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Could not compute temporal saliency")

## 6. Feature Importance Analysis

In [ ]:
# Compute feature importance across multiple samples
feature_saliencies = []

for i in range(n_samples_viz):
    sample = X_test[i:i+1]
    sal = compute_feature_saliency(model, sample)
    if sal is not None:
        feature_saliencies.append(sal[0])

if feature_saliencies:
    feature_saliency_matrix = np.array(feature_saliencies)
    avg_feature_importance = np.mean(feature_saliency_matrix, axis=0)
    
    # Sort by importance
    feature_indices = np.argsort(avg_feature_importance)[::-1]
    
    # Create feature names (customize based on your dataset)
    n_features = len(avg_feature_importance)
    feature_names = [f"Feature {i}" for i in range(n_features)]
    
    # Visualize top features
    top_n = min(15, n_features)
    top_features = feature_indices[:top_n]
    
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.viridis(np.linspace(0, 1, top_n))
    bars = ax.barh(range(top_n), avg_feature_importance[top_features], color=colors)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([feature_names[i] for i in top_features])
    ax.set_xlabel('Importance Score', fontsize=12)
    ax.set_title(f'Top {top_n} Most Important Features', fontsize=14)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    
    # Print ranking
    print(f"\nTop {top_n} Feature Importance Ranking:")
    print("-" * 50)
    for rank, feat_idx in enumerate(top_features, 1):
        print(f"{rank:2d}. {feature_names[feat_idx]:15s} - Score: {avg_feature_importance[feat_idx]:.6f}")
else:
    print("Could not compute feature importance")

## 7. SHAP Values for Feature Attribution (Optional - requires shap library)

If you have SHAP installed, this provides another powerful explainability method

In [ ]:
# Install SHAP if needed
# !pip install shap -q

try:
    import shap
    
    # Create prediction function for SHAP
    def model_predict(X):
        return model.predict(X, verbose=0)
    
    # Use a small background dataset for faster computation
    background_data = X_train[:100]
    test_data = X_test[:10]
    
    print("Computing SHAP values (this may take a minute)...")
    
    # Create explainer
    explainer = shap.DeepExplainer(model, background_data)
    shap_values = explainer.shap_values(test_data)
    
    print(f"SHAP values computed. Shape: {np.array(shap_values).shape}")
    print("\nSHAP analysis ready for visualization")
    
except ImportError:
    print("SHAP library not installed. Install with: pip install shap")
except Exception as e:
    print(f"Note: SHAP computation had an issue: {e}")
    print("This is often due to model complexity. Gradient-based methods above are more reliable.")

## 8. Prediction Confidence & Uncertainty Analysis

In [ ]:
# Generate predictions on test set
try:
    Y_pred = model.predict(X_test, verbose=0)
    
    # Calculate errors
    errors = np.abs(Y_pred.flatten() - Y_test)
    
    # Metrics
    mae = np.mean(errors)
    rmse = np.sqrt(np.mean(errors**2))
    
    print(f"Model Performance on Test Set:")
    print(f"-" * 40)
    print(f"Mean Absolute Error (MAE):  {mae:.4f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
    print(f"Number of samples: {len(Y_test)}")
    
    # Visualize predictions vs actual
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot 1: Predictions vs Actual
    axes[0].scatter(Y_test, Y_pred, alpha=0.6, s=30)
    axes[0].plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()], 'r--', lw=2)
    axes[0].set_xlabel('Actual RUL', fontsize=12)
    axes[0].set_ylabel('Predicted RUL', fontsize=12)
    axes[0].set_title('Predictions vs Actual Values', fontsize=12)
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Error distribution
    axes[1].hist(errors, bins=30, edgecolor='black', alpha=0.7)
    axes[1].axvline(mae, color='red', linestyle='--', linewidth=2, label=f'MAE: {mae:.4f}')
    axes[1].set_xlabel('Absolute Error', fontsize=12)
    axes[1].set_ylabel('Frequency', fontsize=12)
    axes[1].set_title('Error Distribution', fontsize=12)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Error generating predictions: {e}")

## 9. Individual Prediction Explanation

Explain a specific prediction by combining multiple XAI techniques

In [ ]:
def explain_prediction(sample_idx, X_data, Y_data, model):
    """
    Create comprehensive explanation for a single prediction
    """
    # Get sample
    sample = X_data[sample_idx:sample_idx+1]
    actual_rul = Y_data[sample_idx]
    
    # Make prediction
    pred_rul = model.predict(sample, verbose=0)[0, 0]
    error = abs(pred_rul - actual_rul)
    
    # Get explanations
    temporal_sal = compute_temporal_saliency(model, sample)
    feature_sal = compute_feature_saliency(model, sample)
    
    # Create visualization
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)
    
    # 1. Prediction result
    ax1 = fig.add_subplot(gs[0, :])
    ax1.text(0.5, 0.7, f'Sample Index: {sample_idx}', ha='center', fontsize=14, fontweight='bold')
    ax1.text(0.5, 0.4, f'Predicted RUL: {pred_rul:.2f}  |  Actual RUL: {actual_rul:.2f}  |  Error: {error:.2f}', 
            ha='center', fontsize=12)
    ax1.axis('off')
    
    # 2. Temporal saliency
    ax2 = fig.add_subplot(gs[1, 0])
    if temporal_sal is not None:
        ax2.plot(temporal_sal[0], linewidth=2, color='steelblue')
        ax2.fill_between(range(len(temporal_sal[0])), temporal_sal[0], alpha=0.3)
        ax2.set_xlabel('Timestep')
        ax2.set_ylabel('Importance')
        ax2.set_title('Temporal Importance (Saliency)')
        ax2.grid(True, alpha=0.3)
    
    # 3. Feature importance (top 10)
    ax3 = fig.add_subplot(gs[1, 1])
    if feature_sal is not None:
        top_features = np.argsort(feature_sal[0])[-10:][::-1]
        ax3.barh(range(len(top_features)), feature_sal[0][top_features], color='coral')
        ax3.set_yticks(range(len(top_features)))
        ax3.set_yticklabels([f'Feat {i}' for i in top_features])
        ax3.set_xlabel('Importance')
        ax3.set_title('Top 10 Important Features')
        ax3.invert_yaxis()
    
    # 4. Input time series (first 3 features)
    ax4 = fig.add_subplot(gs[2, :])
    for feat_idx in range(min(3, sample.shape[-1])):
        ax4.plot(sample[0, :, feat_idx], label=f'Feature {feat_idx}', linewidth=1.5, alpha=0.8)
    ax4.set_xlabel('Timestep')
    ax4.set_ylabel('Normalized Value')
    ax4.set_title('Input Time Series (First 3 Features)')
    ax4.legend(loc='upper right')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return {
        'sample_idx': sample_idx,
        'predicted_rul': pred_rul,
        'actual_rul': actual_rul,
        'error': error,
        'temporal_saliency': temporal_sal,
        'feature_saliency': feature_sal
    }

# Explain a prediction
try:
    explanation = explain_prediction(5, X_test, Y_test, model)
    print(f"\nPrediction explanation generated successfully!")
except Exception as e:
    print(f"Error: {e}")

## 10. Summary: Key Insights for Model Explainability

In [ ]:
print("\n" + "="*60)
print("XAI ANALYSIS SUMMARY")
print("="*60)

print("\n1. ATTENTION PATTERNS:")
print("   - Analyze which timesteps the transformer focuses on")
print("   - Early timesteps: General degradation trend")
print("   - Late timesteps: Rapid degradation indicators")

print("\n2. TEMPORAL IMPORTANCE:")
print("   - Gradient-based saliency shows critical time periods")
print("   - Peak importance often near end of sequence")
print("   - Indicates model focuses on recent degradation")

print("\n3. FEATURE IMPORTANCE:")
print("   - Top sensors contribute most to RUL predictions")
print("   - Validate against domain knowledge")
print("   - Consider sensor reliability and noise")

print("\n4. PREDICTION CONFIDENCE:")
print("   - Error distribution shows model uncertainty")
print("   - High errors on extreme values (very low/high RUL)")
print("   - Consider confidence intervals for deployment")

print("\n5. RECOMMENDATIONS FOR IMPROVEMENT:")
print("   - Focus on feature engineering for top sensors")
print("   - Use ensemble methods for confidence estimation")
print("   - Monitor attention patterns for distribution shift")
print("   - Implement uncertainty quantification")

print("\n" + "="*60)